# 11. Supplementary analyses (report Sections 4.3 to 4.5)

Rebuilds, from the repository's saved outputs, the analyses that sit around the main fit table:

A. exact versus hybrid fit with a paired bootstrap of the gain over advertisements
B. leave-one-skill-out check of the semantic gain (the AI Agents false match) and halved semantic weight
C. share of uncovered top-30 demand by skill category
D. greedy unit selection

The bootstrap is the slow part. Use `N_BOOT = 200` for a quick look (about 4 minutes) or 1000 for the reported run (about 20 minutes).

In [1]:
from pathlib import Path
import pandas as pd
import supplementary_analyses as sa

ROOT = Path.cwd().resolve().parents[0]
N_BOOT = 200
jobs, job_hits, unit_hits = sa.load(ROOT)
print(len(jobs), 'advertisements,', unit_hits.unit_id.nunique(), 'units')

688 advertisements, 20 units


## A. Exact versus hybrid fit

In [2]:
exact = sa.fit_table(job_hits, jobs, unit_hits, exact_only=True)
hybrid = sa.fit_table(job_hits, jobs, unit_hits)
half = sa.fit_table(job_hits, jobs, unit_hits, semantic_discount=0.5)
base = pd.DataFrame({'exact': exact, 'hybrid': hybrid, 'half_semantic_weight': half})
base['gain'] = base.hybrid - base.exact
base.round(1)

,exact,hybrid,half_semantic_weight,gain
career,,,,
AI Engineer,32.6,40.8,36.7,8.2
Data Analyst,58.1,61.7,59.7,3.6
Data Engineer,59.3,62.9,60.4,3.6
Data Scientist,66.2,68.5,67.5,2.3
ML Engineer,58.2,61.0,59.0,2.8


In [3]:
boot = sa.paired_bootstrap(jobs, job_hits, unit_hits, n_boot=N_BOOT)
boot.round(1)

,career,mean_gain,ci_low,ci_high,excludes_zero
0,AI Engineer,7.5,5.7,8.9,True
1,Data Analyst,4.0,1.3,6.3,True
2,Data Engineer,3.5,1.6,5.4,True
3,Data Scientist,1.7,-0.5,3.6,False
4,ML Engineer,2.7,0.1,4.8,True


## B. Leave-one-skill-out

How many points of each career's hybrid fit come from one semantically added skill?

In [4]:
loo = sa.leave_one_skill_out(jobs, job_hits, unit_hits)
loo[loo.points_from_skill > 0].sort_values('points_from_skill', ascending=False).round(1)

,dropped_skill,career,hybrid,exact,without_skill,points_from_skill
4,AI Agents,AI Engineer,40.8,32.6,33.3,7.5
3,AI Agents,ML Engineer,61.0,58.2,59.0,2.0
1,AI Agents,Data Engineer,62.9,59.3,61.2,1.7
2,AI Agents,Data Analyst,61.7,58.1,60.1,1.6
0,AI Agents,Data Scientist,68.5,66.2,67.0,1.5
10,Predictive Modeling,Data Scientist,68.5,66.2,67.3,1.2


In [5]:
jobs_ai = unit_hits[(unit_hits.skill == 'AI Agents') & (unit_hits.source == 'semantic')][['unit_id', 'score', 'evidence']]
jobs_ai

,unit_id,score,evidence
93,USYD COMP5046 NLP,0.710247,Models - Agents
97,USYD COMP5310 Principles of Data Science,0.651998,Ethical AI usage


## C. Uncovered demand by skill category

In [6]:
sa.uncovered_by_category(jobs, job_hits, unit_hits).round(1)

,skill_category,uncovered_demand,total_demand,share_of_uncovered_pct,covered_pct
0,Machine Learning & AI,2.7,7.0,17.8,61.1
1,Data Visualisation / BI,2.2,2.5,14.4,12.9
2,Data Engineering,1.6,5.6,10.1,72.0
3,Data Analysis & Statistics,1.4,3.1,9.0,55.0
4,Programming,1.3,7.1,8.5,81.6
5,Methods / Architecture,1.3,2.7,8.5,51.4
6,Cloud,1.1,1.1,7.3,0.0
7,NLP / Generative AI,1.1,2.4,7.3,54.4
8,MLOps / Deployment,1.0,1.0,6.7,0.0
9,Development Tools,1.0,1.0,6.4,0.0


## D. Greedy unit selection

In [7]:
greedy = sa.greedy_units(jobs, job_hits, unit_hits)
print(greedy.groupby('career').step.max().rename('units needed'))
greedy.round(1)

career
AI Engineer       3
Data Analyst      6
Data Engineer     4
Data Scientist    6
ML Engineer       6
Name: units needed, dtype: int64


,career,step,unit_added,fit_percent,full_20_unit_fit
0,AI Engineer,1,USYD COMP5046 NLP,13.9,32.6
1,AI Engineer,2,UTS 94693 Big Data Engineering,25.3,32.6
2,AI Engineer,3,UNSW COMP6713 NLP,32.6,32.6
3,Data Analyst,1,UTS 94693 Big Data Engineering,33.6,58.1
4,Data Analyst,2,USYD COMP5310 Principles of Data Science,42.2,58.1
5,Data Analyst,3,UNSW COMP6713 NLP,47.9,58.1
6,Data Analyst,4,USYD DATA5207 Data Analysis,52.9,58.1
7,Data Analyst,5,UTS 36100 Data Science for Innovation,57.0,58.1
8,Data Analyst,6,USYD COMP5046 NLP,58.1,58.1
9,Data Engineer,1,UTS 94693 Big Data Engineering,45.4,59.3
